# Semantic loss: deep learning with symbolic knowledge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/semantic_loss.ipynb)

This notebook implements the core ideas of [*A Semantic Loss Function for Deep Learning with Symbolic Knowledge*](https://arxiv.org/abs/1711.11157) (Xu, Zhang, Friedman, Liang and Van den Broeck, ICML 2018) with `symprog`.

The paper asks how to train a neural network whose outputs must satisfy a logical constraint, such as "exactly one class is true", "the predicted edges form a path" or "the predicted matrix is a ranking". Its answer is the **semantic loss**: read the network's outputs $p$ as independent probabilities of Boolean variables, and penalize the network by how *unlikely* it is that a state sampled from $p$ satisfies the constraint. That probability is a **weighted model count**, which is exactly what `symprog` solvers compute.

This notebook covers:

1. The definition, computed with a `Solver`, and the closed form for the *exactly-one* constraint.
2. The properties the paper proves, checked numerically.
3. The semantic loss as a batched, differentiable PyTorch loss, with `solver.tensorize`.
4. **Semi-supervised classification**: semantic loss on unlabeled data, the paper's headline experiment, on a small synthetic dataset.
5. **Structured outputs**: rankings and paths in a grid.

## Setup

Uncomment the install line if you are running on Colab. If `symprog` is not installed, the notebook uses the version in this repository.

In [1]:
# %pip install -q "symprog[notebooks]"

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

import math
import random
import warnings

import sympy
import torch
from IPython.display import Math, display

from symprog import Solver

## tensorization is experimental; silence its warnings for this notebook
warnings.filterwarnings("ignore", category=symprog.ExperimentalWarning)

# The semantic loss

A constraint $\alpha$ is a Boolean formula over variables $X_1, \dots, X_n$, and a network outputs a vector of probabilities $p = (p_1, \dots, p_n)$, one per variable. The semantic loss is

$$L^s(\alpha, p) = -\log \sum_{x \models \alpha} \; \prod_{i : x \models X_i} p_i \prod_{i : x \models \neg X_i} (1 - p_i).$$

The sum ranges over the states $x$ that satisfy $\alpha$, and each state is weighted by its probability when every $X_i$ is drawn independently with probability $p_i$. So the sum is the probability that a random state drawn from $p$ satisfies $\alpha$: the **weighted model count** of $\alpha$, with $p$ as weights. Its negative logarithm is the `LogProb` inference of a solver:

In [2]:
def constraint_solver(formulas, variables):
    """Compiles the constraints `formulas` over `variables`. The weights are
    placeholders: they are replaced by the network's outputs."""
    solver = Solver()
    solver.add(formulas)
    solver.add_weights({v: 0.5 for v in variables})
    return solver

def semantic_loss(solver, probabilities):
    """The semantic loss of `probabilities` (a dict from variables to probabilities).
    `as_function` gives the inference as a function of the weight vector, ordered as
    `solver.weight_names`."""
    log_prob = solver.as_function(mode="LogProb")
    p = torch.tensor([probabilities[v] for v in solver.weight_names])
    return 0.0 - log_prob(p).item()

The most common constraint is **exactly one** of the output variables being true, e.g., for multi-class classification with one output per class. In `symprog`, `Xor` with several arguments means exactly one. The loss is small for outputs that confidently pick one class, and large when the network picks two classes, or is uncertain:

In [3]:
one_hot = constraint_solver(["Xor(a, b, c)"], ["a", "b", "c"])

examples = {
    "confident, one class": {"a": 0.95, "b": 0.03, "c": 0.02},
    "confident, two classes": {"a": 0.95, "b": 0.90, "c": 0.02},
    "uncertain": {"a": 0.5, "b": 0.5, "c": 0.5},
}
for name, p in examples.items():
    print(f"{name:24} semantic loss = {semantic_loss(one_hot, p):.3f}")

confident, one class     semantic loss = 0.099
confident, two classes   semantic loss = 1.986
uncertain                semantic loss = 0.981


The paper gives a closed form for exactly-one constraints:

$$L^s(\text{exactly one}, p) = -\log \sum_{i=1}^{n} p_i \prod_{j \neq i} (1 - p_j).$$

The solver's symbolic expression (see `symbolic_and_gradients.ipynb`) is the same polynomial, computed from the compiled circuit:

In [4]:
poly = one_hot.query(mode="Prob", symbolic=True).inference
display(Math(r"\sum_{x \models \alpha} \Pr(x) = " + sympy.latex(poly)))

a, b, c = sorted(poly.free_symbols, key=lambda s: s.name)
closed_form = a * (1 - b) * (1 - c) + (1 - a) * b * (1 - c) + (1 - a) * (1 - b) * c
assert sympy.expand(poly - closed_form) == 0

<IPython.core.display.Math object>

For general constraints there is no closed form, and the sum over models can be exponentially large. The paper's key computational idea is to **compile** the constraint into a circuit once, where the loss can be computed in time linear in the size of the circuit. That is what a `symprog` solver does with SDDs.

# Properties

The paper derives the semantic loss from a few axioms, and shows that it is the only function (up to a constant factor) that satisfies them. We can check its main properties numerically.

**A single literal gives the cross-entropy**: for the constraint $X$, the loss is $-\log p$, and for $\neg X$ it is $-\log(1 - p)$. So the semantic loss generalizes the usual loss for a single label:

In [5]:
literal = constraint_solver(["a"], ["a"])
negated = constraint_solver(["Not(a)"], ["a"])

for p in [0.1, 0.5, 0.9]:
    assert math.isclose(semantic_loss(literal, {"a": p}), -math.log(p), rel_tol=1e-6)
    assert math.isclose(semantic_loss(negated, {"a": p}), -math.log(1 - p), rel_tol=1e-6)
    print(f"p = {p}: L(a) = {semantic_loss(literal, {'a': p}):.4f}   -log p = {-math.log(p):.4f}")

p = 0.1: L(a) = 2.3026   -log p = 2.3026
p = 0.5: L(a) = 0.6931   -log p = 0.6931
p = 0.9: L(a) = 0.1054   -log p = 0.1054


**Certain outputs**: when $p$ is a 0/1 vector, the loss is $0$ if the state satisfies the constraint, and infinite otherwise.

In [6]:
print("satisfies the constraint:      ", semantic_loss(one_hot, {"a": 1.0, "b": 0.0, "c": 0.0}))
print("does not satisfy the constraint:", semantic_loss(one_hot, {"a": 1.0, "b": 1.0, "c": 0.0}))

satisfies the constraint:       0.0
does not satisfy the constraint: inf


**Only the meaning of the constraint matters**: logically equivalent constraints have the same loss, however they are written. Below, exactly-one is written in CNF, as "at least one" plus "no two at once".

**Stronger constraints cost more**: if $\alpha$ implies $\beta$, then $L^s(\alpha, p) \geq L^s(\beta, p)$, since every state that satisfies $\alpha$ also satisfies $\beta$. Exactly-one implies at-least-one:

In [7]:
exactly_one_cnf = constraint_solver(
    ["Or(a, b, c)", "Not(And(a, b))", "Not(And(a, c))", "Not(And(b, c))"], ["a", "b", "c"]
)
at_least_one = constraint_solver(["Or(a, b, c)"], ["a", "b", "c"])

rng = random.Random(0)
for _ in range(100):
    p = {v: rng.random() for v in "abc"}
    assert math.isclose(semantic_loss(one_hot, p), semantic_loss(exactly_one_cnf, p), rel_tol=1e-5)
    assert semantic_loss(one_hot, p) >= semantic_loss(at_least_one, p) - 1e-9

p = {"a": 0.6, "b": 0.3, "c": 0.4}
print(f"exactly one: {semantic_loss(one_hot, p):.3f}  (written in CNF: {semantic_loss(exactly_one_cnf, p):.3f})")
print(f"at least one: {semantic_loss(at_least_one, p):.3f}")

exactly one: 0.830  (written in CNF: 0.830)
at least one: 0.184


# A PyTorch loss

To train a network, the loss must be computed for a whole batch of outputs and differentiated. `solver.tensorize(mode="LogProb")` compiles the circuit into a PyTorch module (see `tensorization_and_jax.ipynb`). It is compiled **once**, and then used like any other loss. Its inputs are ordered as `module.input_symbols`.

For 10 classes:

In [8]:
classes = [f"y{i}" for i in range(10)]
exactly_one = constraint_solver([f"Xor({', '.join(classes)})"], classes)
exactly_one_log_prob = exactly_one.tensorize(mode="LogProb")
assert list(exactly_one_log_prob.input_symbols) == classes

def semantic_loss_exactly_one(p):
    """The exactly-one semantic loss for a batch of outputs `p[batch, 10]`."""
    return -exactly_one_log_prob(p)

torch.manual_seed(0)
p = torch.rand(5, 10, requires_grad=True)
loss = semantic_loss_exactly_one(p)

## the closed form, for comparison
closed_form = -torch.log(sum(
    p[:, i] * torch.prod(1 - torch.cat([p[:, :i], p[:, i + 1:]], dim=1), dim=1) for i in range(10)
))
assert torch.allclose(loss, closed_form)
loss

tensor([5.3652, 3.2578, 7.5274, 5.7612, 4.0557], grad_fn=<NegBackward0>)

Gradients flow back to the outputs, and so into the network. Following them moves the outputs towards states that satisfy the constraint. Starting from the first output vector above, where many classes are likely at once, gradient descent on its logits first lowers all outputs, and ends with one class close to 1 (the most likely one) and the others close to 0:

In [9]:
loss.sum().backward()
assert p.grad.shape == p.shape

logits = torch.logit(p[0].detach()).clone().requires_grad_(True)
optimizer = torch.optim.SGD([logits], lr=1.0)
for _ in range(100):
    step_loss = semantic_loss_exactly_one(torch.sigmoid(logits))
    optimizer.zero_grad()
    step_loss.backward()
    optimizer.step()

print("before:", [round(x, 2) for x in p[0].tolist()], f"  loss {semantic_loss_exactly_one(p[0]).item():.3f}")
print("after: ", [round(x, 2) for x in torch.sigmoid(logits).tolist()], f"  loss {step_loss.item():.3f}")

before: [0.5, 0.77, 0.09, 0.13, 0.31, 0.63, 0.49, 0.9, 0.46, 0.63]   loss 5.365
after:  [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.98, 0.01, 0.01]   loss 0.133


# Semi-supervised classification

The paper's main experiment is **semi-supervised** learning: only a few training examples have labels, and the rest are unlabeled. The labeled examples get the usual loss. The unlabeled ones get the semantic loss of the exactly-one constraint: we don't know their class, but we know they have exactly one. This pushes the network to make confident, unambiguous predictions on unlabeled data, which moves the decision boundaries away from dense regions of the data. For exactly-one, the effect resembles entropy minimization; the paper finds the semantic loss slightly better than an entropy regularizer on MNIST, and unlike entropy it applies to any constraint. With as few as 100 labeled images, the paper's results surpassed the previous state of the art on Fashion-MNIST and CIFAR-10, and came close on MNIST.

We reproduce the effect on a small synthetic dataset: 6 well-separated Gaussian clusters, one per class, with only **1 labeled point per class** and 1,794 unlabeled points. Our network has one sigmoid output per class, so its outputs are independent probabilities, as in the definition (the paper uses both sigmoid and softmax outputs).

In [10]:
K = 6
class_names = [f"c{k}" for k in range(K)]
exactly_one_of_K = constraint_solver([f"Xor({', '.join(class_names)})"], class_names).tensorize(mode="LogProb")
assert list(exactly_one_of_K.input_symbols) == class_names

def make_data(seed, n_per_class=300, labeled_per_class=1, spread=0.35):
    """Gaussian clusters around a circle: training points (with a mask of the few
    labeled ones, chosen at random) and a separate test set."""
    g = torch.Generator().manual_seed(seed)
    angles = torch.arange(K) * 2 * math.pi / K
    centers = 2.0 * torch.stack([angles.cos(), angles.sin()], dim=1)

    def sample(n):
        X = torch.cat([centers[k] + spread * torch.randn(n, 2, generator=g) for k in range(K)])
        return X, torch.arange(K).repeat_interleave(n)

    X, y = sample(n_per_class)
    labeled = torch.zeros(len(y), dtype=torch.bool)
    for k in range(K):
        in_class = torch.nonzero(y == k)[:, 0]
        labeled[in_class[torch.randperm(len(in_class), generator=g)[:labeled_per_class]]] = True
    X_test, y_test = sample(500)
    return X, y, labeled, X_test, y_test

def train(X, y, labeled, semantic_weight, seed, steps=1500):
    """Trains a small network on the labeled points (cross-entropy per class), plus the
    semantic loss on the unlabeled points, weighted by `semantic_weight`."""
    torch.manual_seed(seed)
    net = torch.nn.Sequential(
        torch.nn.Linear(2, 64), torch.nn.ReLU(),
        torch.nn.Linear(64, 64), torch.nn.ReLU(),
        torch.nn.Linear(64, K),
    )
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)
    targets = torch.nn.functional.one_hot(y, K).float()
    for _ in range(steps):
        p = torch.sigmoid(net(X))
        loss = torch.nn.functional.binary_cross_entropy(p[labeled], targets[labeled])
        if semantic_weight > 0:
            ## clamp to avoid log(0) for saturated outputs
            unlabeled = p[~labeled].clamp(1e-6, 1 - 1e-6)
            loss = loss + semantic_weight * -exactly_one_of_K(unlabeled).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return net

def accuracy(net, X, y):
    with torch.no_grad():
        return (net(X).argmax(dim=1) == y).float().mean().item()

The result depends on which points happen to be labeled, so we train with and without the semantic loss (with weight $0.1$) on 8 random datasets and compare test accuracy:

In [11]:
results = []
for seed in range(8):
    X, y, labeled, X_test, y_test = make_data(seed)
    baseline = accuracy(train(X, y, labeled, semantic_weight=0.0, seed=seed), X_test, y_test)
    with_semantic_loss = accuracy(train(X, y, labeled, semantic_weight=0.1, seed=seed), X_test, y_test)
    results.append((baseline, with_semantic_loss))
    print(f"dataset {seed}: {int(labeled.sum())} labels   without: {baseline:.3f}   with semantic loss: {with_semantic_loss:.3f}")

mean_without = sum(r[0] for r in results) / len(results)
mean_with = sum(r[1] for r in results) / len(results)
print(f"\nmean accuracy   without: {mean_without:.3f}   with semantic loss: {mean_with:.3f}")
print(f"semantic loss better on {sum(w > b for b, w in results)} of {len(results)} datasets")

dataset 0: 6 labels   without: 0.964   with semantic loss: 0.993


dataset 1: 6 labels   without: 0.949   with semantic loss: 0.993


dataset 2: 6 labels   without: 0.945   with semantic loss: 0.995


dataset 3: 6 labels   without: 0.995   with semantic loss: 0.995


dataset 4: 6 labels   without: 0.960   with semantic loss: 0.995


dataset 5: 6 labels   without: 0.985   with semantic loss: 0.996


dataset 6: 6 labels   without: 0.985   with semantic loss: 0.994


dataset 7: 6 labels   without: 0.982   with semantic loss: 0.994

mean accuracy   without: 0.971   with semantic loss: 0.995
semantic loss better on 8 of 8 datasets


On these clusters, the semantic loss gives a small but consistent gain: with a single label per class, the network without it already does well, and the semantic loss removes most of its remaining errors by keeping the decision boundaries in the gaps between clusters.

The gain is not automatic, though. On harder shapes that we also tried (long parallel stripes, and concentric rings), the same small network did not improve with the semantic loss, whatever its weight, schedule or output layer. In the paper, the weight of the semantic loss is tuned on validation data, and its benefit depends on the data and the model.

# Structured outputs

The semantic loss works for any constraint that can be compiled, not only exactly-one. The paper's other experiments predict **structured objects**, where the outputs must jointly form a valid structure.

#### Rankings

A ranking of $k$ items is a $k \times k$ permutation matrix: variable $r_{ij}$ means "item $i$ is at position $j$", and every row and every column has exactly one true variable. There are $k!$ rankings. The paper's preference-learning task also ranks 4 items (types of sushi, predicted from a user's ranking of 6 others):

In [12]:
k = 4
r = lambda item, position: f"r{item}{position}"
ranking_variables = [r(i, j) for i in range(k) for j in range(k)]

rows = [f"Xor({', '.join(r(i, j) for j in range(k))})" for i in range(k)]
columns = [f"Xor({', '.join(r(i, j) for i in range(k))})" for j in range(k)]
rankings = constraint_solver(rows + columns, ranking_variables)
print("rankings:", int(rankings.query(mode="Count").inference), "=", math.factorial(k))

rankings: 24 = 24


A prediction close to a ranking has a small loss. Putting two items at the same position (and leaving another position empty) has a large loss, even though each row still has exactly one likely position, which a row-by-row loss would not penalize:

In [13]:
## item i at position i, with high probability
ranking = {r(i, j): 0.9 if i == j else 0.03 for i in range(k) for j in range(k)}

## items 0 and 1 both at position 0, and nothing at position 1
clash = dict(ranking)
clash[r(1, 1)], clash[r(1, 0)] = 0.03, 0.9

print(f"close to a ranking:      {semantic_loss(rankings, ranking):.3f}")
print(f"two items at position 0: {semantic_loss(rankings, clash):.3f}")

close to a ranking:      0.787
two items at position 0: 5.760


#### Paths in a grid

In the paper's path experiment, the outputs are the edges of a grid graph, and they must form a **simple path** between two nodes (the paper predicts shortest paths in a $4 \times 4$ grid with some edges removed). We use a $3 \times 3$ grid with 12 edges, and paths from the top-left to the bottom-right corner.

Simple paths are hard to write as a compact formula (connectivity is not a local property). The paper compiles it with a specialized method for simple paths (Nishino et al., 2017). For a small grid we can simply list the paths: the constraint is a disjunction with one term per path, where the path's edges are true and all other edges are false.

In [14]:
nodes = [(row, col) for row in range(3) for col in range(3)]
edges = sorted({
    tuple(sorted([(row, col), (row + dr, col + dc)]))
    for row, col in nodes for dr, dc in [(0, 1), (1, 0)] if (row + dr, col + dc) in nodes
})
edge_name = lambda e: f"e{e[0][0]}{e[0][1]}_{e[1][0]}{e[1][1]}"

def simple_paths(start, end):
    """All simple paths from `start` to `end`, as sets of edges (by depth-first search)."""
    paths = []
    def extend(node, visited, used):
        if node == end:
            paths.append(frozenset(used))
            return
        for e in edges:
            if node in e:
                nxt = e[0] if e[1] == node else e[1]
                if nxt not in visited:
                    extend(nxt, visited | {nxt}, used + [e])
    extend(start, {start}, [])
    return paths

paths = simple_paths((0, 0), (2, 2))
terms = [
    f"And({', '.join(edge_name(e) if e in path else f'Not({edge_name(e)})' for e in edges)})"
    for path in paths
]
grid_paths = constraint_solver([f"Or({', '.join(terms)})"], [edge_name(e) for e in edges])
print(f"{len(paths)} simple paths over {len(edges)} edges; models of the constraint:",
      int(grid_paths.query(mode="Count").inference))

12 simple paths over 12 edges; models of the constraint: 12


A small helper draws a set of edges in the grid:

In [15]:
def draw(edge_set):
    """Draws the grid with the edges in `edge_set` (o: node, ─ and │: edges)."""
    lines = []
    for row in range(3):
        nodes_line = ""
        for col in range(3):
            nodes_line += "o"
            if col < 2:
                nodes_line += "───" if ((row, col), (row, col + 1)) in edge_set else "   "
        lines.append(nodes_line)
        if row < 2:
            lines.append("   ".join("│" if ((row, col), (row + 1, col)) in edge_set else " " for col in range(3)))
    print("\n".join(lines))

draw(paths[0])

o───o───o
        │
o───o───o
│        
o───o───o


Gradient descent on the semantic loss can **repair** a prediction that is not a valid path. Below, a network's (made-up) output gives probability $0.75$ to the edges of one path, but also $0.65$ to two wrong edges, so rounding it does not give a path. We then adjust the outputs to minimize the semantic loss, plus a penalty for moving away from the original prediction:

In [16]:
grid_log_prob = grid_paths.tensorize(mode="LogProb")
order = list(grid_log_prob.input_symbols)
index = {e: order.index(edge_name(e)) for e in edges}

intended = sorted(paths, key=sorted)[3]
prediction = torch.full((len(edges),), 0.15)
for e in intended:
    prediction[index[e]] = 0.75
wrong = [e for e in edges if e not in intended][:2]
for e in wrong:
    prediction[index[e]] = 0.65

def rounded(p):
    return frozenset(e for e in edges if p[index[e]] > 0.5)

print("rounded prediction (a path?", rounded(prediction) in paths, ")")
draw(rounded(prediction))

logits = torch.logit(prediction).clone().requires_grad_(True)
optimizer = torch.optim.Adam([logits], lr=0.1)
for _ in range(200):
    p = torch.sigmoid(logits)
    loss = -grid_log_prob(p) + ((p - prediction) ** 2).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

repaired = rounded(torch.sigmoid(logits))
assert repaired in paths
print("\nafter minimizing the semantic loss (a path?", repaired in paths, ")")
draw(repaired)

rounded prediction (a path? False )
o───o───o
│   │    
o───o   o
│        
o───o───o

after minimizing the semantic loss (a path? True )
o───o   o
    │    
o───o   o
│        
o───o───o


In the paper, the semantic loss is used during *training* in the same way, added to the usual loss. It raised the share of test predictions that are valid paths from 7% to 70% (and of exactly right paths from 6% to 29%), and of valid rankings from 3% to 55%.

# Notes

- **What we reproduced.** The semantic loss itself, its closed form and properties, its use as a PyTorch loss, a small version of the semi-supervised experiment, and the ranking and path constraints. The paper's experiments use MNIST, Fashion-MNIST and CIFAR-10 with larger networks, a $4 \times 4$ grid, and real preference data (sushi rankings).
- **Cost.** Computing the loss takes time linear in the size of the compiled circuit, and the circuit is compiled only once. Some constraints have small circuits (exactly-one, rankings for small $k$), but for others the circuit grows quickly. Listing paths, as above, only works for small grids.
- **Larger circuits.** For large circuits, the KLay backend (`tensorize(mode="LogProb", backend="klay")`, see `tensorization_and_jax.ipynb`) evaluates the circuit layer by layer on the GPU.
- **Numerical stability.** `LogProb` works in log space. Clamping the outputs away from 0 and 1 (as in the training loop above) avoids $\log 0$ when the network saturates.

## Next steps

- **Tensorization** (`tensorization_and_jax.ipynb`): more on compiling inference into PyTorch and JAX.
- **Languages** (`languages.ipynb`): writing constraints as inline programs instead of formula strings.